# LightGBM
Kaggle Team Name: `[7] Material Girls`

Team Members:
- [564323] Eirill Bue
- [544590] Nora Langfeldt Borgenvik
- [586744] Silje Holm Johannesen

## Data Cleaning 

In [1]:
import pandas as pd
import numpy as np
import lightgbm as lgb
import seaborn as sns
from tqdm import tqdm
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import KFold
from sklearn.metrics import mean_tweedie_deviance
from pathlib import Path
import matplotlib.pyplot as plt
from typing import Dict, Any
import optuna
import os, random, numpy as np
os.environ["PYTHONHASHSEED"] = "123"
random.seed(123)
np.random.seed(123)

# Base path for datasets
data_path = "data"

# Load data
receivals = pd.read_csv(f"{data_path}/kernel/receivals.csv")
purchase_orders = pd.read_csv(f"{data_path}/kernel/purchase_orders.csv")
materials = pd.read_csv(f"{data_path}/extended/materials.csv")
transportation = pd.read_csv(f"{data_path}/extended/transportation.csv")

# Convert date columns to datetime
receivals["date_arrival"] = pd.to_datetime(receivals["date_arrival"], utc=True)
purchase_orders["delivery_date"] = pd.to_datetime(purchase_orders["delivery_date"], utc=True)
purchase_orders["created_date_time"] = pd.to_datetime(purchase_orders["created_date_time"], utc=True)
purchase_orders["modified_date_time"] = pd.to_datetime(purchase_orders["modified_date_time"], utc=True)

In [2]:
# Remove duplicates
receivals = receivals.drop_duplicates()
purchase_orders = purchase_orders.drop_duplicates()

# Remove physically impossible values
receivals = receivals[receivals["net_weight"] > 0]
purchase_orders = purchase_orders[purchase_orders["quantity"] > 0]

# Standardize units to kilograms
if "unit" in purchase_orders.columns:
    purchase_orders["unit"] = purchase_orders["unit"].str.lower()
    purchase_orders.loc[purchase_orders["unit"].isin(["pund", "lbs", "pound"]), "quantity"] *= 0.45359237
    purchase_orders["unit"] = "kg"

# Ensure temporal consistency (receivals should not happen before order creation)
merged_temp = receivals.merge(
    purchase_orders[["purchase_order_id", "purchase_order_item_no", "created_date_time"]],
    on=["purchase_order_id", "purchase_order_item_no"],
    how="left"
)
receivals = merged_temp[merged_temp["date_arrival"] >= merged_temp["created_date_time"]].copy()
receivals.drop(columns="created_date_time", inplace=True)

# Keep only valid IDs
receivals = receivals[receivals["purchase_order_id"] > 0]
purchase_orders = purchase_orders[purchase_orders["purchase_order_id"] > 0]

# Merge datasets
training_data = pd.merge(
    receivals,
    purchase_orders,
    on=["purchase_order_id", "purchase_order_item_no"],
    how="inner"
)

# Keep only relevant columns for modeling
training_data = training_data[['rm_id', 'date_arrival', 'net_weight', 'quantity']]

# Floor date to day for aggregation
training_data['date_arrival'] = training_data['date_arrival'].dt.floor('D')

# Aggregate data by material and day
training_data = training_data.groupby(['rm_id', 'date_arrival'], as_index=False).agg(
    net_weight=('net_weight', 'sum'),
    quantity=('quantity', 'first')  # TODO: Check later
)
training_data = training_data.sort_values(['rm_id', 'date_arrival'])

# ---- REMOVE 2020 DATA ----
training_data = training_data[training_data["date_arrival"].dt.year != 2020]
print(f"✅ Removed 2020 rows. Remaining: {len(training_data):,}")

# Save aggregated dataset as training_data.csv
training_data.to_csv(f"{data_path}/cleaned_data.csv", index=False)
print(f"✅ cleaned_data.csv created successfully with {len(training_data):,} rows.")

# Optional: Check for missing values
missing_summary = training_data.isnull().sum()
print(missing_summary)


✅ Removed 2020 rows. Remaining: 38,404
✅ cleaned_data.csv created successfully with 38,404 rows.
rm_id           0
date_arrival    0
net_weight      0
quantity        0
dtype: int64


## Feature Engineering 

In [3]:
# === Cell 2: build daily panel + expanded leakage-safe features ===
data_path = "./data"

training_data = pd.read_csv(f"{data_path}/cleaned_data.csv")
training_data["date_arrival"] = pd.to_datetime(training_data["date_arrival"], utc=True)

# Build full daily history with zeros to 2024-12-31
full_history_list = []
last_history_date = pd.Timestamp("2024-12-31", tz="UTC")
for rm, group in training_data.groupby("rm_id", sort=False):
    start = group["date_arrival"].min().floor("D")
    full_idx = pd.DataFrame({"date_arrival": pd.date_range(start, last_history_date, freq="D", tz="UTC")})
    full_idx["rm_id"] = rm
    merged = full_idx.merge(
        group[["rm_id", "date_arrival", "net_weight"]],
        on=["rm_id", "date_arrival"],
        how="left"
    )
    merged["net_weight"] = merged["net_weight"].fillna(0.0)
    full_history_list.append(merged)

full_history = pd.concat(full_history_list, ignore_index=True).sort_values(["rm_id", "date_arrival"])

def add_features_train(daily_all: pd.DataFrame) -> pd.DataFrame:
    out = []
    ALL_DOWS = [f"dow_{i}" for i in range(7)]

    for rm, g in daily_all.groupby("rm_id", sort=False):
        g = g.sort_values("date_arrival").copy()
        y = g["net_weight"].astype(float)
        y_past = y.shift(1)

        # === Core rolling stats (unchanged) ===
        g["roll90"] = y_past.rolling(90, min_periods=1).mean()
        g["min90"] = y_past.rolling(90, min_periods=1).min().fillna(0.0)
        g["max90"] = y_past.rolling(90, min_periods=1).max().fillna(0.0)
        g["deliveries90"] = (y_past > 0).rolling(90, min_periods=1).sum()
        g["inactive_1y"] = ((y_past > 0).rolling(365, min_periods=1).sum() == 0).astype(int)
        g["5ormore90"] = (g["deliveries90"] >= 5).astype(int)
        g["stddev90"] = y_past.rolling(90, min_periods=2).std().fillna(0.0)

        y_lag1 = y.shift(1)
        y_lag2 = y.shift(2)
        g["covariance90"] = y_lag1.rolling(90, min_periods=2).cov(y_lag2).fillna(0.0)

        # === NEW long-term stability and trend features ===
        g["roll180"] = y_past.rolling(180, min_periods=1).mean().fillna(0.0)
        g["roll365"] = y_past.rolling(365, min_periods=1).mean().fillna(0.0)
        g["trend30"] = y_past.diff(30).fillna(0.0)  # captures direction of change month-to-month

        # === NEW date-based seasonality ===
        g["month"] = g["date_arrival"].dt.month
        g["month_sin"] = np.sin(2 * np.pi * g["month"] / 12)
        g["month_cos"] = np.cos(2 * np.pi * g["month"] / 12)

        g["dayofyear"] = g["date_arrival"].dt.dayofyear
        g["day_sin"] = np.sin(2 * np.pi * g["dayofyear"] / 365)
        g["day_cos"] = np.cos(2 * np.pi * g["dayofyear"] / 365)

        g["year"] = g["date_arrival"].dt.year

        # === Weekday one-hot ===
        dow = pd.Categorical(g["date_arrival"].dt.weekday, categories=list(range(7)))
        dows = pd.get_dummies(dow, prefix="dow").reindex(columns=ALL_DOWS, fill_value=0).reset_index(drop=True)
        g = pd.concat([g.reset_index(drop=True), dows], axis=1)

        out.append(g)

    df = pd.concat(out, ignore_index=True)

    # Safety replacements
    for c in [
        "roll90", "roll180", "roll365", "deliveries90", "stddev90", "covariance90",
        "min90", "max90", "trend30"
    ]:
        df[c] = df[c].replace([np.inf, -np.inf], np.nan).fillna(0.0)

    for c in ["inactive_1y", "5ormore90"] + [f"dow_{i}" for i in range(7)]:
        if c not in df.columns:
            df[c] = 0
        df[c] = df[c].astype(int)

    return df


full_history = add_features_train(full_history)

# Add new features to the list
FEATURES = [
    # rolling and variability
    "roll90","roll180","roll365","deliveries90","inactive_1y","5ormore90",
    "stddev90","covariance90","min90","max90","trend30",
    # seasonality
    "month_sin","month_cos","day_sin","day_cos","year"
] + [f"dow_{i}" for i in range(7)]

training_out = full_history[["rm_id", "date_arrival", "net_weight"] + FEATURES]
training_out.to_csv(f"{data_path}/training_data_lightgbm_date.csv", index=False)
print(f"✅ training_data_lightgbm_date.csv created with {len(training_out):,} rows and features: {', '.join(FEATURES)}")

✅ training_data_lightgbm_date.csv created with 774,860 rows and features: roll90, roll180, roll365, deliveries90, inactive_1y, 5ormore90, stddev90, covariance90, min90, max90, trend30, month_sin, month_cos, day_sin, day_cos, year, dow_0, dow_1, dow_2, dow_3, dow_4, dow_5, dow_6


## The Model 

In [ ]:
# ================================================================
# Unified LightGBM Training + Forecasting + Zero-Active
# ================================================================

from __future__ import annotations
from pathlib import Path
from typing import Dict, Any
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.model_selection import KFold
from sklearn.linear_model import ElasticNetCV
import lightgbm as lgb
from tqdm import tqdm
import holidays
import os, json, random
from datetime import timedelta
from sklearn.metrics import mean_tweedie_deviance

# ---------------- CONFIG ----------------
SEED = 42
np.random.seed(SEED)
random.seed(SEED)
os.environ["PYTHONHASHSEED"] = str(SEED)

# ---------------- Paths ----------------
DATA_PATH = Path("./data")
TRAIN_PATH = DATA_PATH / "training_data_lightgbm_date.csv"
MAP_PATH = DATA_PATH / "prediction_mapping.csv"
SUB_PATH = DATA_PATH / "sample_submission.csv"
OUT_PATH = DATA_PATH / ("lightgbm_submission.csv")

# ---------------- Constants ----------------
K = 3
N_FOLDS = 5
EARLY_STOP = 200

# === Feature list ===
FEATURES = [
    # rolling / activity
    "roll90","roll180","roll365","deliveries90","inactive_1y","5ormore90",
    "stddev90","covariance90","min90","max90","trend30",
    # seasonality
    "month_sin","month_cos","day_sin","day_cos","year",
] + [f"dow_{i}" for i in range(7)]

# ================================================================
# 1. Load and prepare data
# ================================================================
training_data = pd.read_csv(TRAIN_PATH)
training_data["date_arrival"] = pd.to_datetime(training_data["date_arrival"], utc=True)
training_data["rm_id"] = pd.to_numeric(training_data["rm_id"], errors="coerce").astype("Int64")

prediction_mapping = pd.read_csv(MAP_PATH)
prediction_mapping["forecast_start_date"] = pd.to_datetime(prediction_mapping["forecast_start_date"], utc=True)
prediction_mapping["forecast_end_date"] = pd.to_datetime(prediction_mapping["forecast_end_date"], utc=True)
submission = pd.read_csv(SUB_PATH)

# ================================================================
# 2. Cluster assignment 
# ================================================================
per_rm = (
    training_data.groupby("rm_id")
    .agg(
        mean_weight=("net_weight","mean"),
        pct_nonzero=("net_weight", lambda s:(s>0).mean()),
        last_roll90=("roll90", lambda s: float(s.dropna().iloc[-1]) if len(s.dropna())>0 else 0.0)
    ).reset_index()
)
scaler = StandardScaler()
Xc = scaler.fit_transform(per_rm[["mean_weight","pct_nonzero","last_roll90"]])
kmeans = KMeans(n_clusters=K, n_init=20, random_state=SEED)
per_rm["cluster"] = kmeans.fit_predict(Xc)
rm2cluster = dict(zip(per_rm["rm_id"].astype(int), per_rm["cluster"].astype(int)))
print("Cluster sizes:", per_rm["cluster"].value_counts().sort_index().to_dict())

# ================================================================
# 3. Feature selection + hyperparameters (INLINE, no file reads)
# ================================================================

# Selected features from variable selection
sel_payload: Dict[str, Any] = {
    "run_feature_selection": True,
    "use_per_cluster": False,
    "top_k": 12,
    "lasso_filter": False,
    "features_global": [
        "roll90",
        "roll180",
        "dow_5",
        "dow_6",
        "stddev90",
        "deliveries90",
        "year",
        "day_sin",
        "day_cos",
        "roll365",
        "trend30",
        "month_cos"
    ],
    "features_per_cluster": {}
}

# Final training params from hyperparameter tuning 
FINAL_PARAMS: Dict[str, Any] = {
    "objective": "tweedie",
    "metric": "tweedie",
    "n_estimators": 5000,
    "learning_rate": 0.015252697030515176,
    "num_leaves": 45,
    "max_depth": 12,
    "feature_fraction": 0.6298202574719083,
    "bagging_fraction": 0.9947547746402069,
    "bagging_freq": 1,
    "min_child_samples": 145,
    "lambda_l1": 0.013805292809005998,
    "lambda_l2": 2.0386535711370852,
    "tweedie_variance_power": 1.4534286719238088,
    "n_jobs": -1,
    "random_state": 42,
    "verbosity": -1,
    "force_col_wise": True
}

tuned_only: Dict[str, Any] = {
}

# --- Feature selection payload unpacking ---
USE_PER_CLUSTER = bool(sel_payload.get("use_per_cluster", False))
FEATURES_SELECTED_GLOBAL = sel_payload.get("features_global")
selected_by_cluster = {int(k): v for k, v in sel_payload.get("features_per_cluster", {}).items()}

def get_feats_for_cluster(c) -> list[str]:
    if USE_PER_CLUSTER and (c in selected_by_cluster):
        return selected_by_cluster[c]
    return FEATURES_SELECTED_GLOBAL if FEATURES_SELECTED_GLOBAL else FEATURES

print("[INFO] Loaded feature selection from inline dict.")
if USE_PER_CLUSTER and selected_by_cluster:
    print(" - Using per-cluster feature sets (inline).")
else:
    print(f" - Using GLOBAL feature set (inline): {FEATURES_SELECTED_GLOBAL}")

print("\n[FINAL PARAMS] Used for training:")
for k, v in FINAL_PARAMS.items():
    print(f"  {k}: {v}")
print()

def make_lgbm(p):
    return lgb.LGBMRegressor(**p)

def predict_mean(models, X: pd.DataFrame) -> np.ndarray:
    """Average predictions across CV models (using each model's best_iteration_)."""
    preds = [m.predict(X, num_iteration=m.best_iteration_) for m in models]
    return np.mean(np.vstack(preds), axis=0)

def train_kfold_models(X: pd.DataFrame, y: pd.Series, p: Dict[str, Any]):
    """Train LightGBM models with K-Fold CV and early stopping."""
    kf = KFold(n_splits=N_FOLDS, shuffle=False)  # time-aware: no shuffle
    models = []
    for f, (tr, va) in enumerate(kf.split(X), 1):
        m = make_lgbm(p)
        m.fit(
            X.iloc[tr], y.iloc[tr],
            eval_set=[(X.iloc[va], y.iloc[va])],
            eval_metric="tweedie",
            callbacks=[lgb.early_stopping(EARLY_STOP, verbose=0)]
        )
        print(f"  fold {f}/{N_FOLDS} (best_iter={m.best_iteration_})")
        models.append(m)
    return models

# ================================================================
# 4. Train models 
# ================================================================
cluster_models: Dict[Any, list[lgb.LGBMRegressor]] = {}
print("🧠 Training new models...")

for c in range(K):
    rm_ids = set(per_rm.loc[per_rm["cluster"] == c, "rm_id"].astype(int))
    df_c = training_data[training_data["rm_id"].isin(rm_ids)]
    if df_c.empty:
        continue
    feats_c = get_feats_for_cluster(c)
    print(f"Training CV models for cluster {c}: {len(df_c):,} rows | {len(feats_c)} feats")
    cluster_models[c] = train_kfold_models(df_c[feats_c], df_c["net_weight"].astype(float), FINAL_PARAMS)

print(f"Training global fallback on all data: {len(training_data):,} rows")
feats_global = get_feats_for_cluster("global")
cluster_models["global"] = train_kfold_models(training_data[feats_global],
                                              training_data["net_weight"].astype(float), FINAL_PARAMS)

# ================================================================
# 5. Forecasting + Weekend/Holiday redistribution 
# ================================================================
def build_forecast_features(dates: pd.DatetimeIndex, last_vals: Dict[str, float]) -> pd.DataFrame:
    df = pd.DataFrame({"date_arrival": dates})

    carry_keys = [
        "roll90","roll180","roll365","deliveries90","inactive_1y","5ormore90",
        "stddev90","covariance90","min90","max90","trend30",
    ]
    for k in carry_keys:
        df[k] = float(last_vals.get(k, 0.0))

    # seasonality
    month = df["date_arrival"].dt.month
    df["month_sin"] = np.sin(2 * np.pi * month / 12)
    df["month_cos"] = np.cos(2 * np.pi * month / 12)

    dayofyear = df["date_arrival"].dt.dayofyear
    df["day_sin"] = np.sin(2 * np.pi * dayofyear / 365)
    df["day_cos"] = np.cos(2 * np.pi * dayofyear / 365)

    df["year"] = df["date_arrival"].dt.year

    dow = pd.Categorical(df["date_arrival"].dt.weekday, categories=list(range(7)))
    dummies = pd.get_dummies(dow, prefix="dow")
    dummies = dummies.reindex(columns=[f"dow_{i}" for i in range(7)], fill_value=0)
    df = pd.concat([df, dummies], axis=1)

    return df[["date_arrival"] + FEATURES]

CLOSED_MM_DD = {(1, 1), (5, 1)}
def is_closed_day(dt: pd.Timestamp) -> bool:
    return (dt.weekday() >= 5) or ((dt.month, dt.day) in CLOSED_MM_DD)

all_forecasts = []

for _, row in tqdm(prediction_mapping.iterrows(), total=len(prediction_mapping), desc="Forecasting"):
    rm_id = int(row["rm_id"]); ID = row["ID"]
    dates = pd.date_range(row["forecast_start_date"], row["forecast_end_date"], freq="D", tz="UTC")
    c = rm2cluster.get(rm_id)
    models = cluster_models.get(c, cluster_models["global"])
    feats_used = get_feats_for_cluster(c)

    hist = training_data[training_data["rm_id"] == rm_id].sort_values("date_arrival")

    last_keys = [
        "roll90","roll180","roll365","deliveries90","inactive_1y","5ormore90",
        "stddev90","covariance90","min90","max90","trend30",
    ]
    last_vals = {k: float(hist.iloc[-1].get(k, 0)) if len(hist) > 0 else 0 for k in last_keys}

    feats = build_forecast_features(dates, last_vals)
    preds = predict_mean(models, feats[feats_used])

    # Weekend + {Jan 1, May 1}: distribute 3/5 over next 3 open days
    adjusted = []; carry = 0.0; added = 0
    for dt, p in zip(dates, preds):
        if is_closed_day(dt):
            carry += p
            adjusted.append(0.0)
        else:
            if carry > 0 and added < 3:
                p += carry / 5
                added += 1
            adjusted.append(max(p, 0))
    all_forecasts.append(pd.DataFrame({"ID": ID, "rm_id": rm_id, "date_arrival": dates, "net_weight_pred": adjusted}))

forecast_df = pd.concat(all_forecasts, ignore_index=True)
agg = forecast_df.groupby("ID", as_index=False)["net_weight_pred"].sum()
submission = submission[["ID"]].merge(agg, on="ID", how="left")
submission["predicted_weight"] = submission["net_weight_pred"].fillna(0.0)
submission = submission[["ID","predicted_weight"]]
submission.to_csv(OUT_PATH, index=False)
print(f"✅ Forecast saved → {OUT_PATH}")

# ================================================================
# Zero-out by Prophet-style "active set" 
# ================================================================
LAST_DELIVERY_CUTOFF = pd.Timestamp('2024-01-08')
PO_START = pd.Timestamp('2024-10-01', tz='UTC')
PO_END   = pd.Timestamp('2024-12-31', tz='UTC')

RECEIVALS_RAW = DATA_PATH / "kernel" / "receivals.csv"
PO_RAW        = DATA_PATH / "kernel" / "purchase_orders.csv"

td = training_data.copy()
td['date'] = td['date_arrival'].dt.tz_localize(None).dt.floor('D')

# A) Active by last receival date
#activity = td.groupby('rm_id', as_index=False)['date'].agg(last_delivery='max')
activity = (
    td.groupby('rm_id', as_index=False)
    .agg(last_delivery=('date', 'max'))
)
active_by_receival = set(activity.loc[activity['last_delivery'] >= LAST_DELIVERY_CUTOFF, 'rm_id'].astype(int))

# B) Active by POs in window (optional; skipped if files missing)
active_by_pos, pos_error = None, None
try:
    if RECEIVALS_RAW.exists() and PO_RAW.exists():
        receivals_raw = pd.read_csv(RECEIVALS_RAW)
        po_raw = pd.read_csv(PO_RAW)

        receivals_raw['date_arrival'] = pd.to_datetime(receivals_raw['date_arrival'], utc=True)
        po_raw['delivery_date'] = pd.to_datetime(po_raw['delivery_date'], utc=True)

        req_cols_rec = {'purchase_order_id','purchase_order_item_no','date_arrival','rm_id'}
        req_cols_po  = {'purchase_order_id','purchase_order_item_no','delivery_date'}
        if not req_cols_rec.issubset(receivals_raw.columns) or not req_cols_po.issubset(po_raw.columns):
            raise KeyError("Missing required columns for PO-based active rm_ids.")

        last_receivals = (
            receivals_raw
            .sort_values('date_arrival')
            .groupby(['purchase_order_id','purchase_order_item_no'], as_index=False)
            .last()[['purchase_order_id','purchase_order_item_no','rm_id']]
        )
        po_merge = po_raw.merge(last_receivals, on=['purchase_order_id','purchase_order_item_no'], how='left')
        po_window = po_merge.loc[(po_merge['delivery_date'] >= PO_START) & (po_merge['delivery_date'] <= PO_END)]
        active_by_pos = set(po_window['rm_id'].dropna().astype(int).unique().tolist())
    else:
        active_by_pos = None
except Exception as e:
    pos_error = str(e)
    print(f"[INFO] Skipping PO-based active rm_ids due to error: {e}")
    active_by_pos = None

# C) Stability filter on 2024
MIN_DAYS_2024     = 5
MIN_MONTHS_2024   = 3
MAX_MEAN_GAP_DAYS = 45
MAX_TOP_DAY_SHARE = 0.70

df_2024 = td[(td['date'] >= pd.Timestamp('2024-01-01')) &
             (td['date'] <= pd.Timestamp('2024-12-31'))].copy()

days_24 = (df_2024[['rm_id','date']].drop_duplicates()
           .groupby('rm_id', as_index=False)['date'].count()
           .rename(columns={'date':'n_days_2024'}))

def _mean_gap(g):
    d = np.sort(g['date'].unique())
    if len(d) <= 1:
        return np.inf
    gaps = np.diff(d) / np.timedelta64(1, 'D')
    return float(np.mean(gaps))

mean_gap_24 = (df_2024[['rm_id','date']].drop_duplicates()
               .groupby('rm_id', as_index=False)
               .apply(lambda x: pd.Series({'mean_gap_days': _mean_gap(x)}))
               .reset_index(drop=True))

months_24 = (df_2024.assign(month=lambda x: x['date'].dt.month)
             .groupby(['rm_id','month'], as_index=False)['net_weight'].sum()
             .query('net_weight > 0')
             .groupby('rm_id', as_index=False)['month'].count()
             .rename(columns={'month':'months_active_2024'}))

daily_24  = df_2024.groupby(['rm_id','date'], as_index=False)['net_weight'].sum()
totals_24 = daily_24.groupby('rm_id', as_index=False)['net_weight'].sum().rename(columns={'net_weight':'total_2024'})
maxday_24 = daily_24.groupby('rm_id', as_index=False)['net_weight'].max().rename(columns={'net_weight':'max_day_2024'})
spike_24  = (totals_24.merge(maxday_24, on='rm_id', how='left')
             .assign(top_day_share=lambda x: np.where(x['total_2024']>0, x['max_day_2024']/x['total_2024'], 0.0))
             [['rm_id','top_day_share']])

metrics_24 = (days_24
              .merge(mean_gap_24, on='rm_id', how='outer')
              .merge(months_24,  on='rm_id', how='outer')
              .merge(spike_24,   on='rm_id', how='outer')
              .fillna({'n_days_2024':0, 'mean_gap_days':np.inf,
                       'months_active_2024':0, 'top_day_share':1.0}))

stable_mask = (
    (metrics_24['n_days_2024'] >= MIN_DAYS_2024) &
    (metrics_24['months_active_2024'] >= MIN_MONTHS_2024) &
    (metrics_24['mean_gap_days'] <= MAX_MEAN_GAP_DAYS) &
    (metrics_24['top_day_share'] <= MAX_TOP_DAY_SHARE)
)
C_set = set(metrics_24.loc[stable_mask, 'rm_id'].astype(int))

# Base set (A ∩ B) or A if B missing; then ∩ C
if (active_by_pos is not None) and (len(active_by_pos) > 0):
    base_set = set(active_by_receival).intersection(active_by_pos)
    if len(base_set) == 0:
        print("[INFO] PO ∩ Receival base empty; falling back to receival-based (A).")
        base_set = set(active_by_receival)
else:
    print("[INFO] Using receival-based active set only (A).")
    base_set = set(active_by_receival)

FINAL_ACTIVE_SET = set(sorted(base_set.intersection(C_set)))
print(f"[ActiveSet] base={len(base_set)}, stable(C)={len(C_set)} → final active={len(FINAL_ACTIVE_SET)} rm_ids")

# ---- Map rm_id → submission IDs and zero out if NOT active ----
active_ids = set(
    prediction_mapping.loc[prediction_mapping['rm_id'].isin(FINAL_ACTIVE_SET), 'ID']
    .astype(submission['ID'].dtype)
)

sub = pd.read_csv(OUT_PATH)  
sub['active_flag'] = sub['ID'].isin(active_ids).astype(int)
n_zero = int((sub['active_flag'] == 0).sum())
sub.loc[sub['active_flag'] == 0, 'predicted_weight'] = 0.0
sub[['ID','predicted_weight']].to_csv(OUT_PATH, index=False)
print(f"[Zero-Active] Zeroed {n_zero} IDs not in active set. Saved → {OUT_PATH}")

# ================================================================
# 6. Summary 
# ================================================================
print("\n================ LOADED FEATURE SELECTION ================")
if USE_PER_CLUSTER and selected_by_cluster:
    for c in range(K):
        feats_c = get_feats_for_cluster(c)
        print(f"Cluster {c} features ({len(feats_c)}): {feats_c}")
    print(f"Global fallback features ({len(get_feats_for_cluster('global'))}): {get_feats_for_cluster('global')}")
else:
    print(f"Global features used ({len(get_feats_for_cluster('global'))}):")
    print(get_feats_for_cluster('global'))
print("===========================================================\n")


Cluster sizes: {0: 177, 1: 12, 2: 1}
[INFO] Loaded feature selection from inline dict.
 - Using GLOBAL feature set (inline): ['roll90', 'roll180', 'dow_5', 'dow_6', 'stddev90', 'deliveries90', 'year', 'day_sin', 'day_cos', 'roll365', 'trend30', 'month_cos']

[FINAL PARAMS] Used for training:
  objective: tweedie
  metric: tweedie
  n_estimators: 5000
  learning_rate: 0.015252697030515176
  num_leaves: 45
  max_depth: 12
  feature_fraction: 0.6298202574719083
  bagging_fraction: 0.9947547746402069
  bagging_freq: 1
  min_child_samples: 145
  lambda_l1: 0.013805292809005998
  lambda_l2: 2.0386535711370852
  tweedie_variance_power: 1.4534286719238088
  n_jobs: -1
  random_state: 42
  verbosity: -1
  force_col_wise: True

🧠 Training new models...
Training CV models for cluster 0: 739,040 rows | 12 feats
  fold 1/5 (best_iter=430)
  fold 2/5 (best_iter=941)
  fold 3/5 (best_iter=326)
  fold 4/5 (best_iter=237)
  fold 5/5 (best_iter=153)
Training CV models for cluster 1: 31,024 rows | 12 fea

Forecasting: 100%|██████████| 30450/30450 [03:25<00:00, 148.47it/s]


✅ Forecast saved → data/lightgbm_submission.csv
[ActiveSet] base=37, stable(C)=40 → final active=31 rm_ids
[Zero-Active] Zeroed 25800 IDs not in active set. Saved → data/lightgbm_submission.csv

================ LOADED FEATURE SELECTION ================
Global features used (12):
['roll90', 'roll180', 'dow_5', 'dow_6', 'stddev90', 'deliveries90', 'year', 'day_sin', 'day_cos', 'roll365', 'trend30', 'month_cos']



/var/folders/1g/0sfrhqy137s9pstl_0g_bzkh0000gn/T/ipykernel_1899/3092731760.py:345: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: pd.Series({'mean_gap_days': _mean_gap(x)}))
